# Posos — LLM 7B affiné en classifieur (QLoRA) sur GPU Colab

**Avant de lancer :** Exécution → Modifier le type d'exécution → **GPU T4**.

1. Lance la cellule 1 (installation), puis la cellule 2 et **dépose les 5 fichiers** quand le bouton apparaît :
   `input_train.csv`, `output_train.csv`, `input_test.csv`, `correction.csv`, `clean.py`
2. Choisis `MODE` dans la cellule 3 : `"fold0"` (test sur le pli 0 de la validation croisée, ~40 min)
   ou `"full"` (entraînement sur tout + prédiction du test).
3. Exécute tout. À la fin, télécharge le fichier `.npz` produit et dépose-le dans `posos_camembert/colab/`.

In [ ]:
!pip -q install -U "transformers>=4.45,<4.47" "peft>=0.13" "bitsandbytes>=0.44" "accelerate>=0.34" scikit-learn sentencepiece
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import files
import os
os.makedirs("data", exist_ok=True)
for nom in files.upload():
    dest = nom if nom == "clean.py" else f"data/{nom}"
    os.replace(nom, dest)
print(sorted(os.listdir("data")), os.path.exists("clean.py"))

In [ ]:
MODE = "fold0"          # "fold0" pour comparer au pli 0 (bio = 0.8039), "full" pour la soumission
MODELE = "Qwen/Qwen2.5-7B"
EPOCHS = 2
LR = 1e-4
BATCH = 8
MAX_LEN = 192
SEED = 42

In [ ]:
import numpy as np, pandas as pd, torch
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from clean import load_correction_map, make_corrector

# Exactement comme train.py (load_training_frame + --clean + même découpage)
x = pd.read_csv("data/input_train.csv"); y = pd.read_csv("data/output_train.csv")
df = x.merge(y, on="ID"); df["question"] = df["question"].fillna("").astype(str)
corr = make_corrector(load_correction_map(Path("data/correction.csv")))
df["question"] = df["question"].map(corr)
labels_sorted = sorted(df["intention"].unique())
label2id = {int(l): i for i, l in enumerate(labels_sorted)}
df["label"] = df["intention"].map(label2id)

test = pd.read_csv("data/input_test.csv"); test["question"] = test["question"].fillna("").astype(str).map(corr)

if MODE == "fold0":
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    tr_idx, va_idx = list(skf.split(df, df["label"]))[0]
    train_df, val_df = df.iloc[tr_idx], df.iloc[va_idx]
else:
    train_df, val_df = df, None
print(len(train_df), None if val_df is None else len(val_df), len(labels_sorted), "classes")

In [ ]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig,
                          Trainer, TrainingArguments, DataCollatorWithPadding, set_seed)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
set_seed(SEED)

tok = AutoTokenizer.from_pretrained(MODELE)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForSequenceClassification.from_pretrained(
    MODELE, num_labels=len(labels_sorted), quantization_config=bnb, torch_dtype=torch.float16, device_map={"": 0})
model.config.pad_token_id = tok.pad_token_id
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, task_type="SEQ_CLS",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()

def tokenize(b): return tok(b["question"], truncation=True, max_length=MAX_LEN)
ds_tr = Dataset.from_pandas(train_df[["question", "label"]], preserve_index=False).map(tokenize, batched=True)
ds_va = (Dataset.from_pandas(val_df[["question", "label"]], preserve_index=False).map(tokenize, batched=True)
         if val_df is not None else None)
ds_te = Dataset.from_pandas(test[["question"]], preserve_index=False).map(tokenize, batched=True)

In [ ]:
# Perte pondérée par classe (comme --class-weights)
w = torch.tensor(compute_class_weight("balanced", classes=np.arange(len(labels_sorted)),
                                      y=train_df["label"].values), dtype=torch.float)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = torch.nn.functional.cross_entropy(out.logits.float(), labels, weight=w.to(out.logits.device))
        return (loss, out) if return_outputs else loss

args = TrainingArguments(
    output_dir="out", num_train_epochs=EPOCHS, learning_rate=LR,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=16, gradient_accumulation_steps=2,
    warmup_ratio=0.05, lr_scheduler_type="cosine", weight_decay=0.01,
    fp16=True, group_by_length=True, logging_steps=25, save_strategy="no",
    eval_strategy="epoch" if ds_va is not None else "no", report_to="none", seed=SEED)

def metrics(p):
    return {"accuracy": (p.predictions.argmax(-1) == p.label_ids).mean()}

trainer = WeightedTrainer(model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va,
                          data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)
trainer.train()

In [ ]:
# transformers >= 4.46 : group_by_length trie aussi les données en prédiction -> on le coupe
trainer.args.group_by_length = False
from google.colab import files   # au cas où la cellule d'upload a été sautée

def probs(ds):
    logits = trainer.predict(ds).predictions
    return torch.from_numpy(logits).float().softmax(-1).numpy()   # colonne j = intention j

sortie = {"test_probs": probs(ds_te), "test_ids": test["ID"].values}
if val_df is not None:
    vp = probs(ds_va)
    acc = (vp.argmax(-1) == val_df["label"].values).mean()
    print(f"Accuracy pli 0 : {acc:.4f}   (bio 0.8039 · base 0.7852 · v2 0.7877 · av2 0.8045)")
    sortie.update(val_probs=vp, val_ids=val_df["ID"].values)
nom = f"qwen7b_{MODE}.npz"
np.savez(nom, **sortie)
files.download(nom)